# Classical models and chapter submission

Archived from the former Python scripts/package so the Hermes experiments are kept as notebooks.


## Classical TF-IDF Classifiers

Logistic Regression, Linear SVC, and related sparse-feature model code.

_Archived from `src/icd10_coding/models/classifier.py`._


In [ ]:
"""Step 3: logistic-regression classifier cascade."""

import time

import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline

from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR, ensure_output_dirs


MIN_EXAMPLES = 3
RETRIEVAL_CONFIDENCE_THRESHOLD = 0.5


def filter_by_min(df: pd.DataFrame, min_count: int) -> pd.DataFrame:
    counts = df["Code"].value_counts()
    return df[df["Code"].isin(counts[counts >= min_count].index)].copy()


def make_pipeline() -> Pipeline:
    return Pipeline(
        [
            (
                "features",
                FeatureUnion(
                    [
                        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
                        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                    ]
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    solver="saga",
                    C=1.0,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )


def add_classifier_predictions(test, train):
    train_diag_f = filter_by_min(train[train["code_type"] == "diagnosis"], MIN_EXAMPLES)
    train_proc_f = filter_by_min(train[train["code_type"] == "procedure"], MIN_EXAMPLES)

    print("Training diagnosis classifier...")
    clf_diag = make_pipeline()
    clf_diag.fit(train_diag_f["Literal_norm"].values, train_diag_f["Code"].values)

    print("Training procedure classifier...")
    clf_proc = make_pipeline()
    clf_proc.fit(train_proc_f["Literal_norm"].values, train_proc_f["Code"].values)

    print("Training diagnosis-vs-procedure router...")
    router = make_pipeline()
    router.fit(train["Literal_norm"].values, train["code_type"].values)

    test["router_pred"] = router.predict(test["Literal_norm"].values)
    test["clf_code"] = ""
    test["clf_confidence"] = 0.0

    diag_idx = test[test["router_pred"] == "diagnosis"].index
    if len(diag_idx):
        diag_probs = clf_diag.predict_proba(test.loc[diag_idx, "Literal_norm"].values)
        best_diag = diag_probs.argmax(axis=1)
        test.loc[diag_idx, "clf_code"] = clf_diag.named_steps["clf"].classes_[best_diag]
        test.loc[diag_idx, "clf_confidence"] = diag_probs.max(axis=1)

    proc_idx = test[test["router_pred"] == "procedure"].index
    if len(proc_idx):
        proc_probs = clf_proc.predict_proba(test.loc[proc_idx, "Literal_norm"].values)
        best_proc = proc_probs.argmax(axis=1)
        test.loc[proc_idx, "clf_code"] = clf_proc.named_steps["clf"].classes_[best_proc]
        test.loc[proc_idx, "clf_confidence"] = proc_probs.max(axis=1)

    return test


def run() -> None:
    ensure_output_dirs()

    start = time.time()
    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    test = pd.read_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv")
    step2 = pd.read_csv(PREDICTIONS_DIR / "step2_predictions.csv")
    train["Literal_norm"] = train["Literal_norm"].fillna("")
    test["Literal_norm"] = test["Literal_norm"].fillna("")

    test = add_classifier_predictions(test, train)

    train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
    val_split = val_split.copy()
    val_split["Literal_norm"] = val_split["Literal_norm"].fillna("")

    print("Training validation classifiers on the train_split only...")
    val_pred = add_classifier_predictions(val_split, train_split)
    val_pred["s3_correct"] = val_pred["clf_code"] == val_pred["Code"]
    print(f"Step 3 alone accuracy: {val_pred.s3_correct.mean() * 100:.1f}%")

    reference = pd.read_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv")
    reference["Description_norm"] = reference["Description_norm"].fillna("")
    rc_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
    rw_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    rc_ref = rc_vec.fit_transform(reference["Description_norm"])
    rw_ref = rw_vec.fit_transform(reference["Description_norm"])
    v_char = rc_vec.transform(val_pred["Literal_norm"])
    v_word = rw_vec.transform(val_pred["Literal_norm"])
    ret_codes, ret_scores = [], []
    for i in range(0, len(val_pred), 100):
        end = min(i + 100, len(val_pred))
        combined = (cosine_similarity(v_char[i:end], rc_ref) + cosine_similarity(v_word[i:end], rw_ref)) / 2
        best = combined.argmax(axis=1)
        for j in range(end - i):
            bi = best[j]
            ret_codes.append(reference.iloc[bi]["Code"])
            ret_scores.append(float(combined[j, bi]))
        del combined
    val_pred["s2_code"] = ret_codes
    val_pred["s2_score"] = ret_scores

    s1_lookup = {}
    for lit_norm, group in train_split.groupby("Literal_norm"):
        s1_lookup[lit_norm] = group["Code"].value_counts().index[0]
    val_pred["s1_code"] = val_pred["Literal_norm"].map(s1_lookup)
    lookup_keys = list(s1_lookup.keys())
    for idx, row in val_pred[val_pred["s1_code"].isna()].iterrows():
        result = process.extractOne(row["Literal_norm"], lookup_keys, scorer=fuzz.ratio, score_cutoff=60)
        if result is not None:
            val_pred.loc[idx, "s1_code"] = s1_lookup[result[0]]

    val_pred["combined_code"] = val_pred["s1_code"]
    use_retrieval = val_pred["combined_code"].isna() & (
        val_pred["s2_score"] >= RETRIEVAL_CONFIDENCE_THRESHOLD
    )
    val_pred.loc[use_retrieval, "combined_code"] = val_pred.loc[use_retrieval, "s2_code"]
    use_clf = val_pred["combined_code"].isna()
    val_pred.loc[use_clf, "combined_code"] = val_pred.loc[use_clf, "clf_code"]
    val_pred["combined_correct"] = val_pred["combined_code"] == val_pred["Code"]
    print(f"Combined cascade validation accuracy: {val_pred.combined_correct.mean() * 100:.1f}%")

    test = test.merge(
        step2[["id", "step1_code", "step1_method", "retrieval_code", "retrieval_score"]],
        on="id",
        how="left",
    )
    test["final_code"] = test["step1_code"]
    test["final_method"] = test["step1_method"]
    use_ret = test["final_code"].isna() & (test["retrieval_score"] >= RETRIEVAL_CONFIDENCE_THRESHOLD)
    test.loc[use_ret, "final_code"] = test.loc[use_ret, "retrieval_code"]
    test.loc[use_ret, "final_method"] = "retrieval"
    use_clf = test["final_code"].isna()
    test.loc[use_clf, "final_code"] = test.loc[use_clf, "clf_code"]
    test.loc[use_clf, "final_method"] = "classifier"

    detailed_cols = [
        "id", "Literal", "Literal_norm", "router_pred",
        "step1_code", "step1_method", "retrieval_code", "retrieval_score",
        "clf_code", "clf_confidence", "final_code", "final_method",
    ]
    test[detailed_cols].to_csv(PREDICTIONS_DIR / "final_predictions.csv", index=False)
    submission = test[["id", "final_code"]].copy()
    submission.columns = ["id", "Code"]
    submission.to_csv(SUBMISSIONS_DIR / "kaggle_submission.csv", index=False)
    print(f"Saved final_predictions.csv and kaggle_submission.csv in {time.time() - start:.1f}s")


if __name__ == "__main__":
    run()


## Chapter-Level Prediction Model

Code for direct ICD chapter/category prediction.

_Archived from `src/icd10_coding/models/chapter.py`._


In [ ]:
"""Step 5: chapter/category submission model."""

import time

import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR, ensure_output_dirs


def build_features() -> FeatureUnion:
    return FeatureUnion(
        [
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ]
    )


def first_char(value) -> str:
    if pd.isna(value) or value == "":
        return ""
    return str(value)[0]


def predict_chapter(row) -> str:
    if row["chapter_pred"]:
        return row["chapter_pred"]
    if row["s1_chapter"]:
        return row["s1_chapter"]
    if row["retrieval_chapter"]:
        return row["retrieval_chapter"]
    return "null"


def run() -> None:
    ensure_output_dirs()

    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    test = pd.read_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv")
    reference = pd.read_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv")
    step2 = pd.read_csv(PREDICTIONS_DIR / "step2_predictions.csv")

    train["Literal_norm"] = train["Literal_norm"].fillna("")
    test["Literal_norm"] = test["Literal_norm"].fillna("")
    reference["Description_norm"] = reference["Description_norm"].fillna("")
    train["chapter"] = train["Code"].astype(str).str[0]
    reference["chapter"] = reference["Code"].astype(str).str[0]

    print("Training direct chapter classifier...")
    start = time.time()
    vec_chapter = build_features()
    # ICD descriptions are extra labelled examples for the same chapter task.
    chapter_texts = pd.concat([train["Literal_norm"], reference["Description_norm"]], ignore_index=True)
    chapter_labels = pd.concat([train["chapter"], reference["chapter"]], ignore_index=True)
    x_chapter = vec_chapter.fit_transform(chapter_texts)
    clf_chapter = LinearSVC(C=1.0, max_iter=3000, random_state=42)
    clf_chapter.fit(x_chapter, chapter_labels.values)
    print(f"Trained in {time.time() - start:.1f}s")

    x_test = vec_chapter.transform(test["Literal_norm"])
    decisions = clf_chapter.decision_function(x_test)
    test["chapter_pred"] = clf_chapter.classes_[decisions.argmax(axis=1)]
    test["chapter_confidence"] = decisions.max(axis=1)

    test = test.merge(
        step2[["id", "step1_code", "step1_method", "retrieval_code", "retrieval_score"]],
        on="id",
        how="left",
    )
    test["s1_chapter"] = test["step1_code"].apply(first_char)
    test["retrieval_chapter"] = test["retrieval_code"].apply(first_char)
    test["y_category"] = test.apply(predict_chapter, axis=1)

    tr_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
    tr_split = tr_split.copy()
    val_split = val_split.copy()
    tr_split["Literal_norm"] = tr_split["Literal_norm"].fillna("")
    val_split["Literal_norm"] = val_split["Literal_norm"].fillna("")
    val_split["true_chapter"] = val_split["Code"].astype(str).str[0]

    print("Training validation chapter classifier...")
    v_vec = build_features()
    # Use the reference descriptions in validation too; they do not contain leaderboard labels.
    v_texts = pd.concat([tr_split["Literal_norm"], reference["Description_norm"]], ignore_index=True)
    v_labels = pd.concat([tr_split["chapter"], reference["chapter"]], ignore_index=True)
    v_x = v_vec.fit_transform(v_texts)
    v_clf = LinearSVC(C=1.0, max_iter=3000, random_state=42)
    v_clf.fit(v_x, v_labels.values)
    v_decisions = v_clf.decision_function(v_vec.transform(val_split["Literal_norm"]))
    val_split["chapter_pred"] = v_clf.classes_[v_decisions.argmax(axis=1)]
    val_split["chapter_confidence"] = v_decisions.max(axis=1)

    v_lookup = {}
    for lit_norm, group in tr_split.groupby("Literal_norm"):
        v_lookup[lit_norm] = group["Code"].value_counts().index[0]
    val_split["step1_code"] = val_split["Literal_norm"].map(v_lookup)
    val_split["step1_method"] = val_split["step1_code"].apply(
        lambda x: "exact" if pd.notna(x) else "unresolved"
    )
    v_keys = list(v_lookup.keys())
    for idx, row in val_split[val_split["step1_code"].isna()].iterrows():
        result = process.extractOne(row["Literal_norm"], v_keys, scorer=fuzz.ratio, score_cutoff=60)
        if result is not None:
            val_split.loc[idx, "step1_code"] = v_lookup[result[0]]
            val_split.loc[idx, "step1_method"] = "fuzzy"
    val_split["s1_chapter"] = val_split["step1_code"].apply(first_char)

    rcv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
    rwv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    rcr = rcv.fit_transform(reference["Description_norm"])
    rwr = rwv.fit_transform(reference["Description_norm"])
    vc = rcv.transform(val_split["Literal_norm"])
    vw = rwv.transform(val_split["Literal_norm"])
    ret_codes = []
    for i in range(0, len(val_split), 100):
        end = min(i + 100, len(val_split))
        comb = (cosine_similarity(vc[i:end], rcr) + cosine_similarity(vw[i:end], rwr)) / 2
        best = comb.argmax(axis=1)
        for j in range(end - i):
            ret_codes.append(reference.iloc[best[j]]["Code"])
        del comb
    val_split["retrieval_code"] = ret_codes
    val_split["retrieval_chapter"] = val_split["retrieval_code"].apply(first_char)
    val_split["y_category"] = val_split.apply(predict_chapter, axis=1)

    direct_correct = (val_split["chapter_pred"] == val_split["true_chapter"]).sum()
    final_correct = (val_split["y_category"] == val_split["true_chapter"]).sum()
    print(f"Direct chapter classifier accuracy: {direct_correct / len(val_split) * 100:.2f}%")
    print(f"Combined chapter accuracy: {final_correct / len(val_split) * 100:.2f}%")

    submission = test[["id", "Literal", "y_category"]].copy()
    submission["y_category"] = submission["y_category"].fillna("null")
    submission.loc[submission["y_category"] == "", "y_category"] = "null"
    submission.to_csv(SUBMISSIONS_DIR / "kaggle_submission_chapter.csv", index=False)

    detail = test[
        [
            "id", "Literal", "Literal_norm",
            "step1_code", "step1_method", "s1_chapter",
            "retrieval_code", "retrieval_score", "retrieval_chapter",
            "chapter_pred", "chapter_confidence", "y_category",
        ]
    ].copy()
    detail.to_csv(PREDICTIONS_DIR / "final_chapter_predictions.csv", index=False)
    print("Saved chapter submission and detail files")


if __name__ == "__main__":
    run()


## Classifier Experiment Runner

Former command-line wrapper for classical classifier experiments.

_Archived from `scripts/step3_classifier.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.models.classifier import run


if __name__ == "__main__":
    run()


## Chapter Submission Runner

Former command-line wrapper for generating chapter-level submissions.

_Archived from `scripts/step5_chapter_submission.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.models.chapter import run


if __name__ == "__main__":
    run()
